# 10 pi_0 от контроля доступа до полного обучения

            Загрузка pi_0, gated-доступ и нагрузка на видеопамять здесь выше. Сначала по очереди проходим доступ, данные, построение модели и 2-step обратное распространение, и только потом запускаем полное обучение.

            Этот раздел начинается с проверки данных LeRobot и последовательно выполняет 2-step smoke, полное обучение, наблюдение за ресурсами ROCm и поиск checkpoint. Все длительные задачи по умолчанию выключены: сначала проверьте сгенерированную конфигурацию, затем явно включите флаг запуска.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


TOPIC_ROOT = $NOTEBOOK_TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $PROJECT_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT
MODEL_ROOT = $MODEL_ROOT


In [2]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


In [3]:
import shlex

try:
    import yaml
except ImportError as exc:
    raise RuntimeError("В текущем окружении нет PyYAML, сначала выполните pip install pyyaml.") from exc


def require_project_layout():
    required = [
        PROJECT_ROOT / "train_model.py",
        PROJECT_ROOT / "env_config.py",
        PROJECT_ROOT / "asset" / "example_scene_y2.xml",
        PROJECT_ROOT / "mujoco_env" / "y_env2.py",
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "PROJECT_ROOT не является каталогом учебника 04mujoco, отсутствует:\n"
            + "\n".join(str(path) for path in missing)
        )
    return True


def dataset_report(dataset_root):
    dataset_root = Path(dataset_root)
    info_path = dataset_root / "meta" / "info.json"
    tasks_path = dataset_root / "meta" / "tasks.jsonl"
    if not info_path.exists():
        raise FileNotFoundError(f"Не найдены метаданные LeRobot: {info_path}")
    info = json.loads(info_path.read_text(encoding="utf-8"))
    tasks = []
    if tasks_path.exists():
        tasks = [
            json.loads(line)["task"]
            for line in tasks_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    features = info.get("features", {})
    rows = [
        ("repo_id", info.get("repo_id", "")),
        ("episodes", info.get("total_episodes", 0)),
        ("frames", info.get("total_frames", 0)),
        ("fps", info.get("fps", "")),
        ("state shape", features.get("observation.state", {}).get("shape")),
        ("action shape", features.get("action", {}).get("shape")),
        ("tasks", " / ".join(tasks)),
    ]
    md_table(["Элемент данных", "Результат чтения"], rows)
    return info, tasks


def make_training_config(
    policy_type,
    dataset_repo_id,
    dataset_root,
    output_dir,
    steps,
    batch_size,
    chunk_size,
    n_action_steps,
    save_freq,
    seed=42,
):
    return {
        "dataset": {"repo_id": dataset_repo_id, "root": str(Path(dataset_root))},
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "save_checkpoint": True,
        "output_dir": str(Path(output_dir)),
        "batch_size": int(batch_size),
        "job_name": Path(output_dir).name,
        "resume": False,
        "seed": int(seed),
        "num_workers": 4,
        "steps": int(steps),
        "eval_freq": 0,
        "log_freq": max(1, min(50, int(steps))),
        "save_freq": int(save_freq),
        "use_policy_training_preset": True,
        "wandb": {
            "enable": False,
            "project": f"every_embodied_{policy_type}",
            "entity": None,
            "disable_artifact": True,
        },
    }


def write_training_config(name, config):
    config_dir = OUTPUT_ROOT / "configs"
    config_dir.mkdir(parents=True, exist_ok=True)
    path = config_dir / f"{name}.yaml"
    path.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("Конфигурация записана:", path)
    return path


def training_command(config_path):
    return [
        sys.executable,
        str(PROJECT_ROOT / "train_model.py"),
        "--config_path",
        str(Path(config_path)),
    ]


def run_training(config_path, enabled=False):
    require_project_layout()
    command = training_command(config_path)
    print("$", shlex.join(command))
    if not enabled:
        print("Сейчас выводится только предпросмотр команды. После проверки конфигурации переключите соответствующий флаг RUN_* в True.")
        return None
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{PROJECT_ROOT}:{env.get('PYTHONPATH', '')}"
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def show_rocm_resources():
    command = ["rocm-smi", "--showuse", "--showmemuse", "--showtemp"]
    print("$", shlex.join(command))
    if shutil.which(command[0]) is None:
        print("rocm-smi не найден. Проверьте, установлен ли ROCm на этой машине.")
        return
    result = subprocess.run(command, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip(), file=sys.stderr)


## Checkpoint 1: проверка проекта и обучающих данных


In [4]:
require_project_layout()
print("train_model.py =", PROJECT_ROOT / "train_model.py")
print("Обучающие данные =", Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language")))


train_model.py = $PROJECT_ROOT/train_model.py
训练数据 = $TRAIN_DATA_ROOT


In [5]:
candidate = Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language"))
if (candidate / "meta" / "info.json").exists():
    dataset_report(candidate)
else:
    print("Обучающие данные ещё не готовы:", candidate)
    print("Сначала выполните 07_data_collection_and_audit.ipynb или укажите в TRAIN_DATA_ROOT существующий набор данных LeRobot.")


| 数据项 | 读取结果 |
| --- | --- |
| repo_id |  |
| episodes | 20 |
| frames | 2621 |
| fps | 20 |
| state shape | [6] |
| action shape | [7] |
| tasks | Place the blue mug on the plate. / Place the red mug on the plate. |


## Checkpoint 2: проверка доступа Hugging Face (gated)


In [6]:
token_present = bool(os.environ.get("HF_TOKEN"))
print("HF_TOKEN передан в текущий процесс:", token_present)
if not token_present:
    print("Задайте HF_TOKEN в терминале как секрет и заново запустите ядро Jupyter. Не записывайте token в Notebook.")


HF_TOKEN 已注入当前进程： False
请在终端私密设置 HF_TOKEN，再重新启动 Jupyter kernel。不要把 token 写进 Notebook。


Здесь проверяется только наличие переменной окружения, token не выводится. Для обучения pi_0 нужен доступ аккаунта к gated-весам, например PaliGemma.


## Checkpoint 3: генерация конфигурации smoke и полного обучения


In [7]:
MODEL_TYPE = 'pi0'
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language")
TRAIN_DATA_ROOT = Path(
    os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language")
).expanduser()

RUN_SMOKE = False
RUN_FULL_TRAIN = False

SMOKE_OUTPUT = MODEL_ROOT / f"{MODEL_TYPE}_rocm_smoke"
FULL_OUTPUT = MODEL_ROOT / f"{MODEL_TYPE}_rocm_full"

smoke_config = make_training_config(
    policy_type=MODEL_TYPE,
    dataset_repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    output_dir=SMOKE_OUTPUT,
    steps=2,
    batch_size=min(4, 4),
    chunk_size=5,
    n_action_steps=5,
    save_freq=2,
)
full_config = make_training_config(
    policy_type=MODEL_TYPE,
    dataset_repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    output_dir=FULL_OUTPUT,
    steps=20000,
    batch_size=4,
    chunk_size=5,
    n_action_steps=5,
    save_freq=max(1, 20000 // 2),
)

smoke_config_path = write_training_config(f"{MODEL_TYPE}_smoke", smoke_config)
full_config_path = write_training_config(f"{MODEL_TYPE}_full", full_config)
print("smoke output =", SMOKE_OUTPUT)
print("full output =", FULL_OUTPUT)


已写出配置： $OUTPUT_ROOT/configs/pi0_smoke.yaml
已写出配置： $OUTPUT_ROOT/configs/pi0_full.yaml
smoke output = $MODEL_ROOT/pi0_rocm_smoke
full output = $MODEL_ROOT/pi0_rocm_full


Файлы конфигурации записываются в `$OUTPUT_ROOT/configs`, checkpoint — в `$MODEL_ROOT`. Эти каталоги должны находиться на диске с достаточным объёмом, а не в Git-репозитории.


## Checkpoint 4: сначала 2-step smoke


In [8]:
show_rocm_resources()
run_training(smoke_config_path, enabled=RUN_SMOKE)


$ rocm-smi --showuse --showmemuse --showtemp


============================ ROCm System Management Interface ============================
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 31.0
=================================== % time GPU is busy ===================================
GPU[0]		: GPU use (%): 0
=================================== Current Memory Use ===================================
GPU[0]		: GPU Memory Allocated (VRAM%): 9
GPU[0]		: Memory Activity: N/A
GPU[0]		: Not supported on the given system
================================== End of ROCm SMI Log ===================================
$ $NOTEBOOK_PYTHON '$PROJECT_ROOT/train_model.py' --config_path '$OUTPUT_ROOT/configs/pi0_smoke.yaml'
当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。


2-step smoke подтверждает только, что работают загрузка данных, построение модели, forward/backward, шаг оптимизатора и запись checkpoint; он не доказывает сходимость модели и тем более не заменяет успешность closed-loop в MuJoCo.


## Checkpoint 5: запуск полного обучения


In [9]:
run_training(full_config_path, enabled=RUN_FULL_TRAIN)


$ $NOTEBOOK_PYTHON '$PROJECT_ROOT/train_model.py' --config_path '$OUTPUT_ROOT/configs/pi0_full.yaml'
当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。


Снижение loss при обучении pi_0 не гарантирует успешного захвата в замкнутом цикле. Особенно в задачах контакта на малых данных легко возникают проблемы с таймингом gripper и накоплением ошибки. Результаты raw policy, learned auxiliary head и гибрида с scaffold нужно докладывать отдельно.


Типичный порядок восстановления: при нехватке видеопамяти сначала уменьшите `batch_size`; при ошибках DataLoader worker / pickle замените `num_workers` в YAML на `0`; при ошибке `401/403` проверьте права аккаунта и секретный `HF_TOKEN`; если не записывается checkpoint, сначала проверьте диск, на котором находится `$MODEL_ROOT`. Предупреждение базы MIOpen в ROCm можно записать и продолжить наблюдение, если обучение не прерывается; но если ядро завершается, появляются NaN или процесс пропадает, длительное обучение нужно остановить и вернуться к smoke.


## Checkpoint 6: поиск checkpoint и проверка ресурсов


In [10]:
show_rocm_resources()
if (FULL_OUTPUT / "checkpoints").exists():
    checkpoints = sorted((FULL_OUTPUT / "checkpoints").glob("*/pretrained_model"))
    print("Доступные checkpoint:")
    for path in checkpoints:
        print(" -", path)
else:
    print("Полное обучение ещё не запускалось:", FULL_OUTPUT)


$ rocm-smi --showuse --showmemuse --showtemp


============================ ROCm System Management Interface ============================
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 31.0
=================================== % time GPU is busy ===================================
GPU[0]		: GPU use (%): 0
=================================== Current Memory Use ===================================
GPU[0]		: GPU Memory Allocated (VRAM%): 9
GPU[0]		: Memory Activity: N/A
GPU[0]		: Not supported on the given system
================================== End of ROCm SMI Log ===================================
正式训练尚未运行： $MODEL_ROOT/pi0_rocm_full


После обучения не выбирайте только checkpoint с минимальным loss. Следующий шаг — `11_mujoco_closed_loop_deploy.ipynb`: с фиксированным seed, строгим `physical_success` и проверкой по видео сравните промежуточные checkpoint с финальным.


## Снимок логов завершённого обучения


In [11]:
progress = json.loads(
    (ASSET_DIR / "training_progress_snapshot.json").read_text(encoding="utf-8")
)
records = [row for row in progress["records"] if row["model"] == 'pi0']
for row in records:
    ratio = row["current_step"] / max(1, row["total_steps"])
    filled = round(36 * ratio)
    bar = "#" * filled + "." * (36 - filled)
    print(f'{row["model"]:>18} [{bar}] {row["current_step"]}/{row["total_steps"]}')
    print("stage:", row["stage"])
    print("closed-loop physical_success:", row["physical_success"])
    print("note:", row["note"])


               pi0 [####################################] 1500/1500
stage: public raw baseline
closed-loop physical_success: 0/20
note: loss 从 step500 约 0.179 降到 step1500 约 0.046，但 raw closed-loop 没有成功


![Историческая динамика обучения AMD ROCm и результаты замкнутого цикла](../assets/training_progress_overview.png)

                Этот график получен из уже завершённых логов экспериментов на AMD, а не из повторного обучения, которое якобы выполнено при открытии Notebook. Ячейки smoke и full выше дают реальный код запуска; при длительном обучении прогресс в терминале продолжает записываться в выходной каталог, а после завершения решение о пригодности checkpoint принимается по closed-loop оценке из раздела 11.


## Отслеживание фоновых логов обучения в Notebook


In [12]:
def tail_training_log(log_path, lines=25):
    log_path = Path(log_path)
    if not log_path.exists():
        print("Лог обучения ещё не существует:", log_path)
        return
    content = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(content[-lines:]))


TRAIN_LOG = Path(os.environ.get("TRAIN_LOG", FULL_OUTPUT / "train.log"))
print("После запуска обучения эту ячейку можно выполнять повторно: tail_training_log(TRAIN_LOG)")
tail_training_log(TRAIN_LOG)


训练运行后可重复执行：tail_training_log(TRAIN_LOG)
训练日志尚不存在： $MODEL_ROOT/pi0_rocm_full/train.log
